# caltech verisini inceleme

14 dosyayı birleştirip oturumlara bakıyorum sonra bunlardan saatlik doluluk tablosu çıkarıyorum
son kısımda da random forest ile önceki saatin durumunu kullanan basit tahmini karşılaştırıyorum

bir oturum aracın bağlanıp ayrılmasına kadar olan kayıt
bağlanma için connectionTime ayrılma için disconnectTime kullanıyorum
doneChargingTime şarjın bittiği zaman araç o sırada hala bağlı olabilir

önce kütüphaneleri açıp verilerin olduğu klasörü buluyorum
notebooks klasöründeysem bir üst klasöre çıkıyorum
caltech_*.json ile 14 çeyrek dosyasını alıyorum eski tek aylık dosya buraya girmiyor

14 dosya yoksa assert kodu durduruyor
sklearn bulunamadı hatası olursa ayrı hücrede %pip install scikit-learn çalıştırmak gerekiyor

In [1]:
from pathlib import Path
import json
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

data_dir = project_dir / "data" / "raw" / "acn"
json_files=sorted(data_dir.glob("caltech_*.json"))

assert len(json_files) == 14, f"14 dosya olmalı ama {len(json_files)} tane var"
print("dosya sayısı", len(json_files))


dosya sayısı 14


dosyaları sırayla açıp içindeki oturumları aynı listeye ekliyorum
_items oturumların olduğu kısım extend ile bu listedekileri ortak listeye atıyorum
her kayda dosya adını da ekliyorum hangi dosyadan geldiği belli olsun
sonra hepsini pandas tablosuna çeviriyorum

In [2]:
all_sessions=[]

for file_path in json_files:
    with file_path.open(encoding="utf-8") as file:
        quarter_data = json.load(file)

    quarter_sessions = quarter_data["_items"]
    for session in quarter_sessions:
        session["_source_file"] = file_path.name

    all_sessions.extend(quarter_sessions)

sessions=pd.DataFrame(all_sessions)
print("satır sütun", sessions.shape)
print("şarj noktası sayısı",sessions["stationID"].nunique())


satır sütun (31424, 14)
şarj noktası sayısı 55


tarihler yazı olarak geliyor bunları tarih saat olarak çeviriyorum
utc=True ile saatleri utc olarak tutuyorum okunamayan tarih olursa coerce onu NaT yani boş tarih yapıyor

eksik tarih var mı ayrılma zamanı bağlantıdan önce mi veya aynı mı bir de aynı oturum tekrar etmiş mi diye bakıyorum
sorun varsa burada duruyor yoksa zaman sırasına koyup csv olarak kaydediyorum

In [3]:
time_columns=["connectionTime", "disconnectTime", "doneChargingTime"]

for column in time_columns:
    sessions[f"{column}_utc"] = pd.to_datetime(
        sessions[column], utc=True, errors="coerce"
    )

required_times = sessions[["connectionTime_utc", "disconnectTime_utc"]]
missing_times = required_times.isna().any(axis=1)
invalid_intervals=sessions["disconnectTime_utc"] <= sessions["connectionTime_utc"]
duplicate_ids = sessions["sessionID"].duplicated()

print("eksik veya okunamayan tarih",missing_times.sum())
print("hatalı zaman aralığı", invalid_intervals.sum())
print("tekrar eden oturum",duplicate_ids.sum())

assert not missing_times.any(), "bağlantı veya ayrılma zamanı eksik"
assert not invalid_intervals.any(), "hatalı zaman aralığı var"
assert not duplicate_ids.any(), "tekrar eden oturum var"

sessions = sessions.sort_values("connectionTime_utc").reset_index(drop=True)
print("ilk bağlantı utc", sessions["connectionTime_utc"].min())
print("son bağlantı utc",sessions["connectionTime_utc"].max())

output_dir=project_dir / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "caltech_all_sessions.csv"
sessions.to_csv(output_path, index=False)
print("dosya kaydedildi",output_path)


eksik veya okunamayan tarih 0
hatalı zaman aralığı 0
tekrar eden oturum 0
ilk bağlantı utc 2018-04-25 11:08:04+00:00
son bağlantı utc 2021-09-14 01:52:37+00:00


dosya kaydedildi /Users/egearhany/Desktop/elektrikli-arac-sarj-istasyonlarinda-doluluk-tahmini/data/processed/caltech_all_sessions.csv


araç hangi saat başlarında bağlıysa o noktaya 1 yazıyorum
mesela 18:10 da bağlanıp 21:05 te ayrılmışsa 19 20 ve 21 saatlerinde dolu sayıyorum
ceil bir sonraki saat başına yuvarlıyor

pivot_table ile saatler satır şarj noktaları sütun oluyor
aynı saate aynı noktadan iki kayıt gelirse max ile 1 olarak bırakıyorum burada oturumlar çakışıyor mu diye ayrıca bakmıyorum

kayıt olmayan yerlere 0 yazıyorum ama bu kesin boş olduğu anlamına gelmiyor kaynakta kayıt eksik olabilir
55 sütun bütün yıllarda gördüğümüz noktalar hepsinin her tarihte çalıştığını da bilmiyorum

In [4]:
occupied_rows=[]

for session in sessions.itertuples():
    hour = session.connectionTime_utc.ceil("h")
    end_hour= session.disconnectTime_utc.ceil("h")

    while hour < end_hour:
        occupied_rows.append({
            "hour_utc": hour,
            "stationID": session.stationID,
            "occupied": 1
        })
        hour = hour + pd.Timedelta(hours=1)

occupied_hours = pd.DataFrame(occupied_rows)
occupancy_matrix=occupied_hours.pivot_table(
    index="hour_utc",
    columns="stationID",
    values="occupied",
    aggfunc="max"
)

all_hours = pd.date_range(
    start=sessions["connectionTime_utc"].min().floor("h"),
    end=sessions["disconnectTime_utc"].max().ceil("h"),
    freq="h",
    inclusive="left"
)
station_ids = sorted(sessions["stationID"].dropna().unique())
occupancy_matrix = occupancy_matrix.reindex(index=all_hours, columns=station_ids)
occupancy_matrix=occupancy_matrix.fillna(0).astype(int)

occupied_station_count = occupancy_matrix.sum(axis=1)
print("saatlik tablo", occupancy_matrix.shape)
print("ilk saat",occupancy_matrix.index.min())
print("son saat", occupancy_matrix.index.max())
print("en az bir nokta dolu olan saat",(occupied_station_count > 0).sum())
display(occupancy_matrix.head())


saatlik tablo (29705, 55)
ilk saat 2018-04-25 11:00:00+00:00
son saat 2021-09-14 03:00:00+00:00
en az bir nokta dolu olan saat 19791


stationID,2-39-123-23,2-39-123-557,2-39-124-22,2-39-124-558,2-39-125-21,2-39-125-559,2-39-126-20,2-39-126-560,2-39-127-19,2-39-127-561,...,2-39-89-25,2-39-89-439,2-39-90-26,2-39-90-440,2-39-91-437,2-39-91-441,2-39-92-436,2-39-92-442,2-39-95-27,2-39-95-444
2018-04-25 11:00:00+00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2018-04-25 12:00:00+00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2018-04-25 13:00:00+00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2018-04-25 14:00:00+00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2018-04-25 15:00:00+00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,0


ay ay kaç oturum var ve kaç saatte en az bir nokta dolu görünmüş ona bakıyorum
burada ayları los angeles saatine göre alıyorum utc de gece olan saat orada önceki gün olabilir

2020 de sayılar baya düşüyor eylül ve ekimde hiç oturum yok
kullanım mı azalmış kayıt mı eksik sadece buradan anlayamıyorum

In [5]:
local_connections=sessions["connectionTime_utc"].dt.tz_convert("America/Los_Angeles")
session_months = sessions.groupby(local_connections.dt.strftime("%Y-%m")).size()
session_months.name = "session_count"

local_hours = occupancy_matrix.index.tz_convert("America/Los_Angeles")
active_hours = occupied_station_count > 0
active_hour_months=active_hours.groupby(local_hours.strftime("%Y-%m")).sum()
active_hour_months.name = "hours_with_any_occupancy"

monthly_summary=pd.concat([session_months, active_hour_months], axis=1)
monthly_summary = monthly_summary.fillna(0).astype(int).sort_index()
monthly_summary.index.name = "month"
display(monthly_summary)


,session_count,hours_with_any_occupancy
month,,
2018-04,270,139
2018-05,1776,744
2018-06,1845,720
2018-07,2053,744
2018-08,2430,744
2018-09,2315,720
2018-10,2424,744
2018-11,1309,708
2018-12,877,727


X modele verdiğim bilgiler y ise tahmin etmesini istediğim doluluk
saat ve gün bilgisine önceki saati ve 24 saat öncesini ekliyorum
shift(1) bir satır shift(24) de 24 satır geriye kaydırıyor ilk 24 satırın geçmişi eksik onları çıkarıyorum

hedef o saatin doluluğu elimdeki en yeni doluluk bir saat öncesi yani bir saat sonrasını tahmin ediyorum
24 saat farkını utc üzerinden alıyorum yaz saati değiştiğinde yerel saatte tam aynı saate denk gelmeyebilir

In [6]:
calendar_features=pd.DataFrame(index=occupancy_matrix.index)
calendar_features["hour_local"] = local_hours.hour
calendar_features["day_of_week_local"] = local_hours.dayofweek
calendar_features["is_weekend_local"]=local_hours.dayofweek >= 5

lag_1 = occupancy_matrix.shift(1).add_prefix("previous_hour_")
lag_24=occupancy_matrix.shift(24).add_prefix("previous_day_")

X = pd.concat([calendar_features, lag_1, lag_24], axis=1)
y=occupancy_matrix.copy()
X = X.iloc[24:]
y = y.iloc[24:]

print("giriş tablosu", X.shape)
print("hedef tablosu",y.shape)


giriş tablosu (29681, 113)
hedef tablosu (29681, 55)


ilk yüzde 80 ile modeli eğitiyorum kalan yüzde 20 ile test ediyorum
sıralamayı karıştırmıyorum eski veriden öğrenip sonraki saatleri tahmin etsin diye

şimdilik ayrı doğrulama verisi ayırmadım test sonuçlarına da birkaç kez baktım
bundan sonraki ayar denemeleri için eğitim kısmından doğrulama verisi ayırmam gerekiyor

In [7]:
split_index=int(len(X) * 0.8)
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train=y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("eğitim dönemi",X_train.index.min(),"ile",X_train.index.max())
print("test dönemi", X_test.index.min(), "ile", X_test.index.max())


eğitim dönemi 2018-04-26 11:00:00+00:00 ile 2021-01-09 18:00:00+00:00
test dönemi 2021-01-09 19:00:00+00:00 ile 2021-09-14 03:00:00+00:00


her nokta için random forest eğitiyorum MultiOutputClassifier 55 sütunu ayrı ayrı öğreniyor
bir noktayı tahmin ederken diğer noktaların geçmiş bilgilerini de kullanabiliyor

ayarlar ilk denemeyle aynı 30 ağaç en fazla 12 seviye yaprakta en az 5 satır
class_weight dolu ve boş örneklerin sayısındaki farkı hesaba katıyor random_state=42 de tekrar çalıştırınca aynı sonucu almak için
fit eğitim yapıyor predict ile de test tahminlerini alıyorum

In [8]:
base_model=RandomForestClassifier(
    n_estimators=30,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)
model = MultiOutputClassifier(base_model)
model.fit(X_train, y_train)
predictions = model.predict(X_test)


bir de önceki saat doluysa sonraki saat de doludur diye basit bir tahmin yapıyorum buna baseline diyorum
araçlar birkaç saat bağlı kaldığı için bu yöntem iyi sonuç verebiliyor

iki yöntemi aynı test verisinde karşılaştırıyorum ravel ile bütün nokta saat değerlerini tek diziye açıyorum
accuracy doğru tahmin oranı precision dolu dediklerimin ne kadarının doğru olduğu recall ise dolu saatlerin ne kadarını bulduğum
f1 precision ve recall u birlikte değerlendiriyor boş örnek çok o yüzden sadece accuracy ye bakmıyorum

In [9]:
previous_hour_columns=list(lag_1.columns)
baseline_predictions = X_test[previous_hour_columns].to_numpy()
actual_values = y_test.to_numpy().ravel()

prediction_sets = {
    "önceki saat baseline": baseline_predictions,
    "random forest": predictions
}

for name, predicted_values in prediction_sets.items():
    predicted_values = predicted_values.ravel()
    print(name)
    print("accuracy",accuracy_score(actual_values,predicted_values))
    print("precision", precision_score(actual_values, predicted_values, zero_division=0))
    print("recall",recall_score(actual_values,predicted_values,zero_division=0))
    print("f1", f1_score(actual_values, predicted_values, zero_division=0))
    print()


önceki saat baseline
accuracy 0.9849418898433552
precision 0.7332899305555556
recall 0.7332103721384399
f1 0.7332501491889546

random forest
accuracy 0.9806881345031926
precision 0.6313509611045934
recall 0.7590322230660735
f1 0.6893289979308306



sonuçları ay ay da ayırıyorum burada test verisindeki utc aylarını kullanıyorum
yukarıda yerel saate göre ayırmıştım ay sınırındaki bazı saatler farklı aya girebilir

actual_occupancy_rate o ayki nokta saat değerlerinin ne kadarının 1 olduğu
bütün noktalar her tarihte çalışıyor muydu bilmiyorum bu yüzden bunu sahanın kesin doluluk oranı diye almıyorum
ocak ve eylül de testte tam ay değil

In [10]:
monthly_results=[]
test_months = y_test.index.strftime("%Y-%m")

for month in test_months.unique():
    month_mask= test_months == month
    actual_month = y_test.loc[month_mask].to_numpy().ravel()
    model_month = predictions[month_mask].ravel()
    baseline_month = baseline_predictions[month_mask].ravel()

    monthly_results.append({
        "month": month,
        "actual_occupancy_rate": actual_month.mean(),
        "baseline_f1": f1_score(actual_month, baseline_month, zero_division=0),
        "random_forest_f1": f1_score(actual_month, model_month, zero_division=0),
        "baseline_recall": recall_score(actual_month, baseline_month, zero_division=0),
        "random_forest_recall": recall_score(actual_month, model_month, zero_division=0)
    })

monthly_results=pd.DataFrame(monthly_results).set_index("month")
display(monthly_results.round(3))


,actual_occupancy_rate,baseline_f1,random_forest_f1,baseline_recall,random_forest_recall
month,,,,,
2021-01,0.010,0.644,0.517,0.643,0.753
2021-02,0.015,0.720,0.618,0.721,0.772
2021-03,0.019,0.742,0.668,0.740,0.777
2021-04,0.021,0.705,0.666,0.706,0.753
2021-05,0.024,0.712,0.678,0.713,0.740
2021-06,0.030,0.711,0.695,0.710,0.737
2021-07,0.037,0.743,0.702,0.744,0.749
2021-08,0.048,0.745,0.731,0.744,0.754
2021-09,0.066,0.784,0.739,0.787,0.807


bu sefer noktaları tek tek karşılaştırıyorum column_index tahmin tablosundan o noktanın sütununu seçiyor
iki yöntemin f1 değerini hesaplayıp farkını alıyorum
f1_difference pozitifse random forest negatifse baseline daha iyi

bazı noktalarda çok az dolu saat var bunlarda birkaç tahmin bile skoru çok değiştiriyor

In [11]:
station_results=[]

for column_index, station_id in enumerate(y_test.columns):
    actual_station = y_test[station_id].to_numpy()
    model_station= predictions[:, column_index]
    baseline_station = baseline_predictions[:, column_index]

    baseline_f1 = f1_score(actual_station, baseline_station, zero_division=0)
    model_f1= f1_score(actual_station, model_station, zero_division=0)

    station_results.append({
        "stationID": station_id,
        "actual_occupied_hours": int(actual_station.sum()),
        "baseline_f1": baseline_f1,
        "random_forest_f1": model_f1,
        "f1_difference": model_f1 - baseline_f1,
        "baseline_recall": recall_score(actual_station, baseline_station, zero_division=0),
        "random_forest_recall": recall_score(actual_station, model_station, zero_division=0)
    })

station_results = pd.DataFrame(station_results)
station_results=station_results.sort_values("f1_difference", ascending=False)
display(station_results.round(3))


,stationID,actual_occupied_hours,baseline_f1,random_forest_f1,f1_difference,baseline_recall,random_forest_recall
6,2-39-126-20,95,0.653,0.660,0.007,0.653,0.653
38,2-39-81-4550,841,0.281,0.286,0.005,0.281,0.558
54,2-39-95-444,3,0.667,0.667,0.000,0.667,0.667
13,2-39-129-563,0,0.000,0.000,0.000,0.000,0.000
1,2-39-123-557,0,0.000,0.000,0.000,0.000,0.000
21,2-39-139-567,1,0.000,0.000,0.000,0.000,0.000
39,2-39-82-384,1,0.000,0.000,0.000,0.000,0.000
19,2-39-138-566,1,0.000,0.000,0.000,0.000,0.000
40,2-39-82-385,1,0.000,0.000,0.000,0.000,0.000
17,2-39-131-565,0,0.000,0.000,0.000,0.000,0.000


testte hiç dolu saat olmayan noktaları bu f1 karşılaştırmasından çıkarıyorum
ikisinin de f1 i sıfır çıkınca hangisi daha iyi anlaşılmıyor
kalanlarda kaç noktada kim daha iyi sayıyorum

son iki tabloda farkın en yüksek ve en düşük olduğu noktalar var
üstte eşit kalanlar da olabilir hepsi random forest daha iyi demek değil

In [12]:
evaluated_stations=station_results[
    station_results["actual_occupied_hours"] > 0
].copy()

rf_wins = (evaluated_stations["f1_difference"] > 0).sum()
ties = (evaluated_stations["f1_difference"] == 0).sum()
baseline_wins=(evaluated_stations["f1_difference"] < 0).sum()

print("testte dolu saat olan nokta",len(evaluated_stations))
print("random forest'ın kazandığı nokta:", rf_wins)
print("eşit",ties)
print("baseline daha iyi", baseline_wins)

print("f1 farkı en yüksek 10 nokta")
display(evaluated_stations.head(10).round(3))
print("f1 farkı en düşük 10 nokta")
display(evaluated_stations.tail(10).round(3))


testte dolu saat olan nokta 45
random forest'ın kazandığı nokta: 2
eşit 11
baseline daha iyi 32
f1 farkı en yüksek 10 nokta


,stationID,actual_occupied_hours,baseline_f1,random_forest_f1,f1_difference,baseline_recall,random_forest_recall
6,2-39-126-20,95,0.653,0.660,0.007,0.653,0.653
38,2-39-81-4550,841,0.281,0.286,0.005,0.281,0.558
54,2-39-95-444,3,0.667,0.667,0.000,0.667,0.667
21,2-39-139-567,1,0.000,0.000,0.000,0.000,0.000
39,2-39-82-384,1,0.000,0.000,0.000,0.000,0.000
19,2-39-138-566,1,0.000,0.000,0.000,0.000,0.000
40,2-39-82-385,1,0.000,0.000,0.000,0.000,0.000
41,2-39-83-386,7,0.429,0.429,0.000,0.429,0.429
26,2-39-78-364,14,0.786,0.786,0.000,0.786,0.786
25,2-39-78-363,59,0.864,0.864,0.000,0.864,0.864


f1 farkı en düşük 10 nokta


,stationID,actual_occupied_hours,baseline_f1,random_forest_f1,f1_difference,baseline_recall,random_forest_recall
52,2-39-92-442,5,0.800,0.727,-0.073,0.800,0.800
0,2-39-123-23,74,0.730,0.655,-0.074,0.730,0.784
22,2-39-78-360,19,0.789,0.714,-0.075,0.789,0.789
32,2-39-79-378,58,0.828,0.750,-0.078,0.828,0.828
28,2-39-78-366,112,0.839,0.757,-0.082,0.839,0.848
53,2-39-95-27,62,0.790,0.689,-0.101,0.790,0.823
51,2-39-92-436,85,0.612,0.439,-0.173,0.612,0.318
36,2-39-79-382,13,0.769,0.588,-0.181,0.769,0.769
42,2-39-83-387,2,0.500,0.000,-0.500,0.500,0.000
9,2-39-127-561,3,0.667,0.000,-0.667,0.667,0.000


önceki saati kullanan yöntemin f1 i yaklaşık 0,733 random forest 0,689 çıktı
random forest daha fazla dolu saat bulmuş ama yanlış dolu tahminleri de artmış
45 noktanın 32 sinde baseline iyi 2 sinde random forest iyi 11 inde eşit

bu ayarlarla baseline ı geçemedim
sonraki denemede eğitimden doğrulama kısmı ayırıp önceki haftanın doluluğu gibi bilgiler ekleyebilirim
kayıt olmayan saatleri ve noktaların hangi dönemlerde çalıştığını da ayrıca kontrol etmek gerekiyor